# LeetCode 3772 — Maximum Subgraph Score in a Tree bằng Python

- Link: https://leetcode.com/problems/maximum-subgraph-score-in-a-tree/
- Độ khó: **Hard**
- Chủ đề: Cây, DP trên cây, **rerooting** (đổi gốc), DFS/BFS lặp, "chỉ lấy đóng góp dương"

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxSubgraphScore(self, n: int, edges: List[List[int]], good: List[int]) -> List[int]:
```

## 1. Mục tiêu học

1. Đổi bài toán về **trọng số đỉnh** `w[v] = +1` (tốt) / `−1` (xấu): điểm của một đồ thị con liên thông = tổng trọng số.
2. Viết DP xuống `down[u] = w[u] + Σ max(0, down[c])`: cây con liên thông tốt nhất **có gốc tại `u`** trong cây con của `u`.
3. Dùng **rerooting** để có đáp án cho **mọi** đỉnh trong `O(n)`: `ans[c] = down[c] + max(0, ans[u] − max(0, down[c]))`.
4. Cài đặt **không đệ quy** (BFS order) để an toàn với `n = 10^5`.
5. Kiểm chứng bằng vét cạn mọi tập con liên thông trên cây nhỏ.

## 2. Đề bài & ví dụ

Cho cây `n` đỉnh, mảng `good` (1 = tốt, 0 = xấu). Điểm của đồ thị con = số đỉnh tốt − số đỉnh xấu. Với mỗi đỉnh `i`, tìm điểm **lớn nhất** trong các đồ thị con **liên thông chứa `i`**.

**Ví dụ 1:** `n = 3, edges = [[0,1],[1,2]], good = [1,0,1]` → `[1,1,1]`.

**Ví dụ 2:** `n = 5, edges = [[1,0],[1,2],[1,3],[3,4]], good = [0,1,0,1,1]` → `[2,3,2,3,3]`.

**Ví dụ 3:** `n = 2, edges = [[0,1]], good = [0,0]` → `[-1,-1]`.

**Ràng buộc:** `2 ≤ n ≤ 10^5`.

In [1]:
from typing import List
from collections import deque
import random, time

examples = [
    (3, [[0, 1], [1, 2]], [1, 0, 1], [1, 1, 1]),
    (5, [[1, 0], [1, 2], [1, 3], [3, 4]], [0, 1, 0, 1, 1], [2, 3, 2, 3, 3]),
    (2, [[0, 1]], [0, 0], [-1, -1]),
]
for n, e, g, exp in examples:
    print(n, e, g, "-> kỳ vọng", exp)

3 [[0, 1], [1, 2]] [1, 0, 1] -> kỳ vọng [1, 1, 1]
5 [[1, 0], [1, 2], [1, 3], [3, 4]] [0, 1, 0, 1, 1] -> kỳ vọng [2, 3, 2, 3, 3]
2 [[0, 1]] [0, 0] -> kỳ vọng [-1, -1]


## 3. Kiến thức nền

### 3.1. Danh sách kề và thứ tự BFS

Với cây, BFS từ gốc cho thứ tự mà **cha luôn đứng trước con**. Duyệt ngược thứ tự đó ⇒ con được xử lý trước cha (thay cho DFS hậu thứ tự, tránh giới hạn đệ quy).

In [2]:
def bfs_order(n, edges, root=0):
    adj = [[] for _ in range(n)]
    for a, b in edges:
        adj[a].append(b); adj[b].append(a)
    parent = [-1] * n; order = [root]; seen = [False] * n; seen[root] = True
    for u in order:                       # list mở rộng trong lúc duyệt = BFS
        for v in adj[u]:
            if not seen[v]:
                seen[v] = True; parent[v] = u; order.append(v)
    return adj, parent, order

adj, par, order = bfs_order(5, [[1, 0], [1, 2], [1, 3], [3, 4]])
print("thứ tự BFS:", order, "| cha:", par)

thứ tự BFS: [0, 1, 2, 3, 4] | cha: [-1, 0, 1, 1, 3]


### 3.2. "Chỉ lấy phần dương"

Đồ thị con liên thông chứa `u` (trong cây con của `u`) = `u` + với mỗi con `c`, **hoặc không lấy gì**, **hoặc** một đồ thị con liên thông chứa `c`. Các lựa chọn độc lập ⇒ với mỗi con, lấy `max(0, tốt_nhất(c))`.

In [3]:
w = [1 if x else -1 for x in [0, 1, 0, 1, 1]]
print("trọng số w:", w)

trọng số w: [-1, 1, -1, 1, 1]


### 3.3. Rerooting (đổi gốc)

Sau khi có `down[u]` cho gốc 0, đáp án cho gốc là `down[0]`. Khi chuyển gốc từ `u` sang con `c`, phần "phía trên" của `c` chính là đáp án của `u` **trừ đi** đóng góp của `c` vào `u`: `ans[u] − max(0, down[c])`. Nó được thêm vào `c` nếu dương.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Với mỗi đỉnh `i`, đáp án = đồ thị con liên thông tốt nhất khi **coi `i` là gốc**: `i` cộng phần dương từ mỗi nhánh kề. Nếu làm lại DP cho từng gốc ⇒ `O(n²)`. Rerooting tận dụng việc các nhánh "dưới" không đổi khi gốc di chuyển, chỉ nhánh "trên" thay đổi.

Ví dụ 2 (gốc 0): `w = [-1, 1, -1, 1, 1]`.

- `down[4] = 1`, `down[3] = 1 + 1 = 2`, `down[2] = −1`, `down[1] = 1 + max(0,−1) + 2 = 3`, `down[0] = −1 + 3 = 2`.
- `ans[0] = 2`; `ans[1] = 3 + max(0, 2 − 3) = 3`; `ans[2] = −1 + max(0, 3 − 0) = 2`; `ans[3] = 2 + max(0, 3 − 2) = 3`; `ans[4] = 1 + max(0, 3 − 1) = 3`. ✔

## 5. Cách 1 — Brute Force: vét cạn tập con liên thông

Duyệt mọi tập con `mask` (`n ≤ 12`), kiểm tra liên thông, cập nhật đáp án cho mọi đỉnh trong `mask`.

In [4]:
def brute(n, edges, good):
    adj = [[] for _ in range(n)]
    for a, b in edges:
        adj[a].append(b); adj[b].append(a)
    w = [1 if x else -1 for x in good]
    ans = [-10**9] * n
    for mask in range(1, 1 << n):
        start = (mask & -mask).bit_length() - 1
        seen = 1 << start; stack = [start]
        while stack:
            u = stack.pop()
            for v in adj[u]:
                if mask >> v & 1 and not seen >> v & 1:
                    seen |= 1 << v; stack.append(v)
        if seen != mask:
            continue
        score = sum(w[v] for v in range(n) if mask >> v & 1)
        for v in range(n):
            if mask >> v & 1 and score > ans[v]:
                ans[v] = score
    return ans

for n, e, g, exp in examples:
    print(brute(n, e, g), "kỳ vọng", exp)

[1, 1, 1] kỳ vọng [1, 1, 1]
[2, 3, 2, 3, 3] kỳ vọng [2, 3, 2, 3, 3]
[-1, -1] kỳ vọng [-1, -1]


**Nút thắt:** `O(2^n · n)` — chỉ dùng để kiểm chứng.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 (DP xuống):** `down[u] = w[u] + Σ_{c con u} max(0, down[c])`.

**Bước 2 (DP lên / rerooting):** `ans[root] = down[root]`, và với cạnh cha `u` → con `c`:

```
up_part = ans[u] − max(0, down[c])     # tốt nhất chứa u nhưng KHÔNG đi vào nhánh c
ans[c]  = down[c] + max(0, up_part)
```

Vì sao `ans[u] − max(0, down[c])` là "tốt nhất chứa `u`, không vào nhánh `c`": `ans[u]` = `w[u]` + tổng phần dương của **mọi** nhánh kề `u` (kể cả nhánh cha); bỏ đúng phần của nhánh `c` là xong — vì các nhánh độc lập.

## 7. Các cách tiếp theo

- **Cách 2:** DP lại cho **từng gốc** — `O(n²)`.
- **Cách 3:** rerooting với công thức `ans[c] = down[c] + max(0, ans[u] − max(0, down[c]))` — `O(n)` (lời giải nộp).
- **Cách 4:** rerooting với mảng `up[]` tường minh: `up[c] = max(0, w[u] + up[u] + Σ_{c' ≠ c} max(0, down[c']))`.

### 7.1. Cách 2 — DP cho từng gốc

In [5]:
def per_root(n, edges, good):
    w = [1 if x else -1 for x in good]
    res = []
    for r in range(n):
        adj, parent, order = bfs_order(n, edges, r)
        down = w[:]
        for u in reversed(order):
            p = parent[u]
            if p >= 0 and down[u] > 0:
                down[p] += down[u]
        res.append(down[r])
    return res

for n, e, g, exp in examples:
    print(per_root(n, e, g), "kỳ vọng", exp)

[1, 1, 1] kỳ vọng [1, 1, 1]
[2, 3, 2, 3, 3] kỳ vọng [2, 3, 2, 3, 3]
[-1, -1] kỳ vọng [-1, -1]


**Độ phức tạp:** `O(n²)` — với `n = 10^5` là `10^10`, quá chậm, nhưng là kiểm chứng tốt cho cây vừa (`n ≈ 300`).

### 7.2. Cách 3 — Rerooting (lời giải chính)

In [6]:
def reroot(n, edges, good):
    adj, parent, order = bfs_order(n, edges, 0)
    w = [1 if x else -1 for x in good]
    down = w[:]
    for u in reversed(order):                   # con trước cha
        p = parent[u]
        if p >= 0 and down[u] > 0:
            down[p] += down[u]
    ans = [0] * n
    ans[0] = down[0]
    for u in order[1:]:                          # cha trước con
        p = parent[u]
        ans[u] = down[u] + max(0, ans[p] - max(0, down[u]))
    return ans

for n, e, g, exp in examples:
    print(reroot(n, e, g), "kỳ vọng", exp)

[1, 1, 1] kỳ vọng [1, 1, 1]
[2, 3, 2, 3, 3] kỳ vọng [2, 3, 2, 3, 3]
[-1, -1] kỳ vọng [-1, -1]


### 7.3. Cách 4 — Mảng `up[]` tường minh

`up[c]` = đóng góp (đã cắt về `≥ 0`) của phần cây **phía trên** `c` khi `c` là gốc. Khi đó `ans[c] = down[c] + up[c]`. Công thức: `up[c] = max(0, ans[p] − max(0, down[c]))` — tương đương Cách 3, nhưng tách bạch "trên" và "dưới" giúp mở rộng cho các bài rerooting khác (khi phép gộp không có nghịch đảo, ta dùng tiền tố/hậu tố trên danh sách con).

In [7]:
def reroot_prefix_suffix(n, edges, good):
    # Không dùng phép trừ: với mỗi u, tính tổng phần dương của các con trừ một con bằng tiền tố/hậu tố
    adj, parent, order = bfs_order(n, edges, 0)
    w = [1 if x else -1 for x in good]
    children = [[] for _ in range(n)]
    for u in order[1:]:
        children[parent[u]].append(u)
    down = w[:]
    for u in reversed(order):
        for c in children[u]:
            down[u] += max(0, down[c])
    up = [0] * n
    for u in order:
        ch = children[u]; k = len(ch)
        vals = [max(0, down[c]) for c in ch]
        pre = [0] * (k + 1); suf = [0] * (k + 1)
        for i in range(k): pre[i + 1] = pre[i] + vals[i]
        for i in range(k - 1, -1, -1): suf[i] = suf[i + 1] + vals[i]
        for i, c in enumerate(ch):
            up[c] = max(0, w[u] + up[u] + pre[i] + suf[i + 1])
    return [down[v] + up[v] for v in range(n)]

for n, e, g, exp in examples:
    print(reroot_prefix_suffix(n, e, g), "kỳ vọng", exp)

[1, 1, 1] kỳ vọng [1, 1, 1]
[2, 3, 2, 3, 3] kỳ vọng [2, 3, 2, 3, 3]
[-1, -1] kỳ vọng [-1, -1]


## 8. TRACE từng bước

In [8]:
def trace_reroot(n, edges, good):
    adj, parent, order = bfs_order(n, edges, 0)
    w = [1 if x else -1 for x in good]
    print("w =", w, "| thứ tự BFS:", order, "| cha:", parent)
    down = w[:]
    print("— Pha 1: DP xuống (duyệt ngược BFS)")
    for u in reversed(order):
        p = parent[u]
        if p >= 0:
            add = max(0, down[u])
            down[p] += add
            print(f"  down[{u}] = {down[u]:>2} -> cộng {add} vào down[{p}] = {down[p]}")
    ans = [0] * n; ans[0] = down[0]
    print("— Pha 2: đổi gốc (duyệt xuôi BFS); ans[0] =", ans[0])
    for u in order[1:]:
        p = parent[u]
        up_part = ans[p] - max(0, down[u])
        ans[u] = down[u] + max(0, up_part)
        print(f"  ans[{u}] = down[{u}]={down[u]} + max(0, ans[{p}]={ans[p]} - {max(0, down[u])}) = {ans[u]}")
    print("đáp án:", ans)

trace_reroot(5, [[1, 0], [1, 2], [1, 3], [3, 4]], [0, 1, 0, 1, 1])

w = [-1, 1, -1, 1, 1] | thứ tự BFS: [0, 1, 2, 3, 4] | cha: [-1, 0, 1, 1, 3]
— Pha 1: DP xuống (duyệt ngược BFS)
  down[4] =  1 -> cộng 1 vào down[3] = 2
  down[3] =  2 -> cộng 2 vào down[1] = 3
  down[2] = -1 -> cộng 0 vào down[1] = 3
  down[1] =  3 -> cộng 3 vào down[0] = 2
— Pha 2: đổi gốc (duyệt xuôi BFS); ans[0] = 2
  ans[1] = down[1]=3 + max(0, ans[0]=2 - 3) = 3
  ans[2] = down[2]=-1 + max(0, ans[1]=3 - 0) = 2
  ans[3] = down[3]=2 + max(0, ans[1]=3 - 2) = 3
  ans[4] = down[4]=1 + max(0, ans[3]=3 - 1) = 3
đáp án: [2, 3, 2, 3, 3]


In [9]:
trace_reroot(6, [[0, 1], [0, 2], [2, 3], [2, 4], [4, 5]], [0, 0, 1, 0, 1, 1])
print("brute:", brute(6, [[0, 1], [0, 2], [2, 3], [2, 4], [4, 5]], [0, 0, 1, 0, 1, 1]))

w = [-1, -1, 1, -1, 1, 1] | thứ tự BFS: [0, 1, 2, 3, 4, 5] | cha: [-1, 0, 0, 2, 2, 4]
— Pha 1: DP xuống (duyệt ngược BFS)
  down[5] =  1 -> cộng 1 vào down[4] = 2
  down[4] =  2 -> cộng 2 vào down[2] = 3
  down[3] = -1 -> cộng 0 vào down[2] = 3
  down[2] =  3 -> cộng 3 vào down[0] = 2
  down[1] = -1 -> cộng 0 vào down[0] = 2
— Pha 2: đổi gốc (duyệt xuôi BFS); ans[0] = 2
  ans[1] = down[1]=-1 + max(0, ans[0]=2 - 0) = 1
  ans[2] = down[2]=3 + max(0, ans[0]=2 - 3) = 3
  ans[3] = down[3]=-1 + max(0, ans[2]=3 - 0) = 2
  ans[4] = down[4]=2 + max(0, ans[2]=3 - 2) = 3
  ans[5] = down[5]=1 + max(0, ans[4]=3 - 1) = 3
đáp án: [2, 1, 3, 2, 3, 3]
brute: [2, 1, 3, 2, 3, 3]


## 9. Code nộp LeetCode

In [10]:
class Solution:
    def maxSubgraphScore(self, n: int, edges: List[List[int]], good: List[int]) -> List[int]:
        adj = [[] for _ in range(n)]
        for a, b in edges:
            adj[a].append(b); adj[b].append(a)
        parent = [-1] * n
        order = [0]
        seen = [False] * n; seen[0] = True
        for u in order:
            for v in adj[u]:
                if not seen[v]:
                    seen[v] = True; parent[v] = u; order.append(v)
        down = [1 if x else -1 for x in good]
        for u in reversed(order):
            p = parent[u]
            if p >= 0 and down[u] > 0:
                down[p] += down[u]
        ans = [0] * n
        ans[0] = down[0]
        for u in order[1:]:
            p = parent[u]
            ans[u] = down[u] + max(0, ans[p] - max(0, down[u]))
        return ans

sol = Solution()
for n, e, g, exp in examples:
    got = sol.maxSubgraphScore(n, e, g)
    print(got, "✔" if got == exp else "✘")

[1, 1, 1] ✔
[2, 3, 2, 3, 3] ✔
[-1, -1] ✔


## 10. Vì sao thuật toán đúng

1. **DP xuống.** Mọi đồ thị con liên thông `S` chứa `u` và nằm trong cây con của `u` phân rã duy nhất thành `{u}` ∪ (với mỗi con `c`: `S ∩ subtree(c)`, là rỗng hoặc liên thông chứa `c`). Các phần độc lập ⇒ tối ưu từng phần ⇒ `down[u] = w[u] + Σ max(0, down[c])` (quy nạp theo chiều cao).
2. **Gốc.** Với gốc `0`, mọi đồ thị con liên thông chứa `0` nằm trong cây con của `0` ⇒ `ans[0] = down[0]`.
3. **Chuyển gốc.** Giả sử `ans[p]` đúng. Khi coi `p` là gốc, `ans[p] = w[p] + Σ_{nhánh kề p} max(0, best(nhánh))`, trong đó nhánh chứa `c` có `best = down[c]`. Vậy `ans[p] − max(0, down[c])` = tốt nhất chứa `p` không đi vào nhánh `c` = `best` của nhánh chứa `p` khi `c` là gốc. Áp dụng phân rã ở bước 1 cho gốc `c`: `ans[c] = down[c] + max(0, ans[p] − max(0, down[c]))`.
4. Quy nạp theo thứ tự BFS ⇒ mọi `ans` đúng.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(2^n · n)` | `O(n)` |
| 2. DP từng gốc | `O(n²)` | `O(n)` |
| 3. Rerooting | `O(n)` | `O(n)` |
| 4. Rerooting tiền tố/hậu tố | `O(n)` | `O(n)` |

## 12. Bộ test

In [11]:
FAST = [per_root, reroot, reroot_prefix_suffix, lambda n, e, g: Solution().maxSubgraphScore(n, e, g)]

for n, e, g, exp in examples:
    assert brute(n, e, g) == exp and all(f(n, e, g) == exp for f in FAST)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [12]:
# 12.2 ca biên
assert Solution().maxSubgraphScore(2, [[0, 1]], [1, 1]) == [2, 2]
assert Solution().maxSubgraphScore(2, [[0, 1]], [1, 0]) == [1, 0]
star = [[0, i] for i in range(1, 6)]
assert Solution().maxSubgraphScore(6, star, [0, 1, 1, 0, 1, 0]) == brute(6, star, [0, 1, 1, 0, 1, 0])
print("Ca biên: OK;", Solution().maxSubgraphScore(6, star, [0, 1, 1, 0, 1, 0]))

Ca biên: OK; [2, 2, 2, 1, 2, 1]


In [13]:
# 12.3 ngẫu nhiên: so với vét cạn
def rand_tree(rng, n):
    return [[i, rng.randrange(i)] for i in range(1, n)]

rng = random.Random(3772)
for _ in range(800):
    n = rng.randint(2, 11)
    e = rand_tree(rng, n); g = [rng.randint(0, 1) for _ in range(n)]
    b = brute(n, e, g)
    assert all(f(n, e, g) == b for f in FAST), (n, e, g)
print("800 cây ngẫu nhiên (vét cạn): OK")

800 cây ngẫu nhiên (vét cạn): OK


In [14]:
# 12.4 test lớn: đường thẳng (sâu nhất) và ngẫu nhiên, n = 10^5
for name, e in [("đường thẳng", [[i, i + 1] for i in range(10**5 - 1)]), ("ngẫu nhiên", rand_tree(random.Random(1), 10**5))]:
    g = [random.Random(2).randint(0, 1) for _ in range(10**5)]
    t0 = time.time(); a = Solution().maxSubgraphScore(10**5, e, g); t1 = time.time()
    print(name, "| khớp tiền tố/hậu tố:", a == reroot_prefix_suffix(10**5, e, g), "| thời gian:", round(t1 - t0, 2), "s")

đường thẳng | khớp tiền tố/hậu tố: True | thời gian: 0.04 s


ngẫu nhiên | khớp tiền tố/hậu tố: True | thời gian: 0.1 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ dùng `down[i]` làm đáp án (quên phần phía trên)

In [15]:
def wrong_down_only(n, edges, good):
    adj, parent, order = bfs_order(n, edges, 0)
    down = [1 if x else -1 for x in good]
    for u in reversed(order):
        if parent[u] >= 0 and down[u] > 0: down[parent[u]] += down[u]
    return down

n, e, g, exp = examples[1]
print("sai:", wrong_down_only(n, e, g), "| đúng:", exp)

sai: [2, 3, -1, 2, 1] | đúng: [2, 3, 2, 3, 3]


### Lỗi 2 — Không trừ đóng góp của chính `c` khi đổi gốc (đếm hai lần)

In [16]:
def wrong_no_subtract(n, edges, good):
    adj, parent, order = bfs_order(n, edges, 0)
    down = [1 if x else -1 for x in good]
    for u in reversed(order):
        if parent[u] >= 0 and down[u] > 0: down[parent[u]] += down[u]
    ans = [0] * n; ans[0] = down[0]
    for u in order[1:]:
        ans[u] = down[u] + max(0, ans[parent[u]])               # SAI
    return ans

print("sai:", wrong_no_subtract(n, e, g), "| đúng:", exp)

sai: [2, 5, 4, 7, 8] | đúng: [2, 3, 2, 3, 3]


### Lỗi 3 — Trừ `down[c]` thay vì `max(0, down[c])`

Nếu `down[c] < 0`, nhánh `c` **không** được tính vào `ans[p]`; trừ `down[c]` âm sẽ cộng thêm điểm ảo.

In [17]:
def wrong_subtract_raw(n, edges, good):
    adj, parent, order = bfs_order(n, edges, 0)
    down = [1 if x else -1 for x in good]
    for u in reversed(order):
        if parent[u] >= 0 and down[u] > 0: down[parent[u]] += down[u]
    ans = [0] * n; ans[0] = down[0]
    for u in order[1:]:
        ans[u] = down[u] + max(0, ans[parent[u]] - down[u])     # SAI
    return ans

print("sai:", wrong_subtract_raw(n, e, g), "| đúng:", exp)

sai: [2, 3, 3, 3, 3] | đúng: [2, 3, 2, 3, 3]


### Lỗi 4 — DFS đệ quy với `n = 10^5`

Cây đường thẳng sâu `10^5` vượt giới hạn đệ quy mặc định (1000) của Python ⇒ `RecursionError`. Dùng thứ tự BFS như trên (hoặc tăng giới hạn + chạy trong thread có stack lớn — rủi ro hơn).

In [18]:
import sys
def dfs_down(u, p, adj, down):
    for v in adj[u]:
        if v != p:
            dfs_down(v, u, adj, down)
            down[u] += max(0, down[v])

N = 5000
adj_line = [[] for _ in range(N)]
for i in range(N - 1):
    adj_line[i].append(i + 1); adj_line[i + 1].append(i)
try:
    dfs_down(0, -1, adj_line, [1] * N)
    print("không lỗi (giới hạn đệ quy đã được nâng?)", sys.getrecursionlimit())
except RecursionError:
    print("RecursionError với đường thẳng", N, "đỉnh — giới hạn mặc định:", sys.getrecursionlimit())

RecursionError với đường thẳng 5000 đỉnh — giới hạn mặc định: 1000


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Với **mỗi** đỉnh, tối ưu một đại lượng trên cây" ⇒ **rerooting**: một pha xuống, một pha lên.
- Đồ thị con liên thông tối đa tổng trọng số ⇒ `down[u] = w[u] + Σ max(0, down[c])` (giống *Maximum Subarray* trên cây).
- Khi gộp có phép nghịch đảo (`+`) ⇒ trừ trực tiếp; khi không có (`max`, `gcd`) ⇒ tiền tố/hậu tố trên danh sách con.
- Python + cây lớn ⇒ luôn dùng **thứ tự BFS** thay cho đệ quy.

## 15. Biến thể của bài

1. **Trọng số tuỳ ý** (`w[v]` nguyên bất kỳ): công thức không đổi.
2. **Chỉ hỏi tối ưu toàn cục** (không cần mỗi đỉnh): `max(down)` đủ — không cần rerooting.
3. **Giới hạn kích thước đồ thị con ≤ K**: DP knapsack trên cây `O(nK)`.
4. **Tổng khoảng cách tới mọi đỉnh** (LC 834): rerooting kinh điển khác.

In [19]:
# Biến thể 1: trọng số tuỳ ý — chỉ cần đổi cách khởi tạo
def reroot_weighted(n, edges, w):
    adj, parent, order = bfs_order(n, edges, 0)
    down = w[:]
    for u in reversed(order):
        if parent[u] >= 0 and down[u] > 0: down[parent[u]] += down[u]
    ans = [0] * n; ans[0] = down[0]
    for u in order[1:]:
        ans[u] = down[u] + max(0, ans[parent[u]] - max(0, down[u]))
    return ans

def brute_weighted(n, edges, w):
    adj = [[] for _ in range(n)]
    for a, b in edges: adj[a].append(b); adj[b].append(a)
    best = [-10**18] * n
    for mask in range(1, 1 << n):
        st = (mask & -mask).bit_length() - 1; seen = 1 << st; stack = [st]
        while stack:
            u = stack.pop()
            for v in adj[u]:
                if mask >> v & 1 and not seen >> v & 1: seen |= 1 << v; stack.append(v)
        if seen != mask: continue
        sc = sum(w[v] for v in range(n) if mask >> v & 1)
        for v in range(n):
            if mask >> v & 1: best[v] = max(best[v], sc)
    return best

rng = random.Random(15)
for _ in range(400):
    n = rng.randint(2, 10); e = rand_tree(rng, n); w = [rng.randint(-5, 5) for _ in range(n)]
    assert reroot_weighted(n, e, w) == brute_weighted(n, e, w)
print("Biến thể 1 (trọng số tuỳ ý): OK")

Biến thể 1 (trọng số tuỳ ý): OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Mọi tập con liên thông | `O(2^n n)` | `O(n)` | Kiểm chứng |
| 2. DP từng gốc | Lặp lại DP xuống | `O(n²)` | `O(n)` | Cây vừa |
| 3. Rerooting (trừ) | `ans[p] − max(0, down[c])` | `O(n)` | `O(n)` | **Lời giải chuẩn** |
| 4. Rerooting (tiền tố/hậu tố) | Không cần phép trừ | `O(n)` | `O(n)` | Phép gộp không khả nghịch |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `n = 4`, đường thẳng `0-1-2-3`, `good = [1,0,0,1]`.

**Bài 2 (tính tay).** Hình sao tâm `0` với 4 lá, `good = [0,1,1,0,0]`.

**Bài 3 (điền TODO).** Viết `best_global(n, edges, good)` trả về điểm lớn nhất trên **mọi** đồ thị con liên thông.

**Bài 4 (tự viết lại).** Viết lại lời giải bằng DFS lặp dùng stack tường minh (không dùng BFS).

**Bài 5 (mini challenge).** Với mỗi đỉnh `i`, đếm **số** đồ thị con liên thông chứa `i` (mod `10^9+7`) bằng rerooting (gợi ý: `f[u] = Π (1 + f[c])`; phép gộp nhân, chia có thể bằng 0 mod p ⇒ dùng tiền tố/hậu tố).

In [20]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def best_global(n, edges, good):
    # TODO: max(down) sau pha 1
    pass

In [21]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def count_connected_per_node(n, edges):
    # TODO: rerooting với tích, dùng tiền tố/hậu tố
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Chứa `0`: chỉ `{0}` = 1 hoặc cả đường = 0 ⇒ 1. (2) Chứa `1`: `{0,1}` = 0. (3) Đáp án `[1,0,0,1]`.

**Bài 2.** (1) `w = [-1,1,1,-1,-1]`. (2) Chứa `0`: `−1 + 1 + 1 = 1`. (3) Lá tốt: `1 + max(0, 1 − 1) = 1`; lá xấu: `−1 + 1 = 0` ⇒ `[1,1,1,0,0]`.

**Bài 3.** (1) Đồ thị con tối ưu có một đỉnh "cao nhất" `u`. (2) Khi đó điểm = `down[u]`. (3) `max(down)`.

**Bài 4.** (1) Stack chứa `(u, cha, đã_mở_rộng)`. (2) Lần pop thứ hai mới gộp con. (3) Pha 2 tương tự với thứ tự tiền thứ tự.

**Bài 5.** (1) `f[u] = Π(1 + f[c])` đếm đồ thị con chứa `u` trong cây con. (2) `g[c] = 1 + (tích các nhánh khác của p, gồm cả phía trên)`. (3) `ans[c] = f[c] · g[c]`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2
assert Solution().maxSubgraphScore(4, [[0, 1], [1, 2], [2, 3]], [1, 0, 0, 1]) == [1, 0, 0, 1] == brute(4, [[0, 1], [1, 2], [2, 3]], [1, 0, 0, 1])
star4 = [[0, i] for i in range(1, 5)]
assert Solution().maxSubgraphScore(5, star4, [0, 1, 1, 0, 0]) == [1, 1, 1, 0, 0] == brute(5, star4, [0, 1, 1, 0, 0])
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [23]:
# Bài 3, 4
def best_global_sol(n, edges, good):
    adj, parent, order = bfs_order(n, edges, 0)
    down = [1 if x else -1 for x in good]
    for u in reversed(order):
        if parent[u] >= 0 and down[u] > 0: down[parent[u]] += down[u]
    return max(down)

def dfs_stack_solution(n, edges, good):
    adj = [[] for _ in range(n)]
    for a, b in edges: adj[a].append(b); adj[b].append(a)
    w = [1 if x else -1 for x in good]; down = w[:]; parent = [-1] * n; pre = []
    stack = [(0, -1, False)]
    while stack:
        u, p, done = stack.pop()
        if done:
            for v in adj[u]:
                if v != p: down[u] += max(0, down[v])
            continue
        parent[u] = p; pre.append(u); stack.append((u, p, True))
        for v in adj[u]:
            if v != p: stack.append((v, u, False))
    ans = [0] * n; ans[0] = down[0]
    for u in pre[1:]:
        ans[u] = down[u] + max(0, ans[parent[u]] - max(0, down[u]))
    return ans

rng = random.Random(4)
for _ in range(400):
    n = rng.randint(2, 10); e = rand_tree(rng, n); g = [rng.randint(0, 1) for _ in range(n)]
    b = brute(n, e, g)
    assert best_global_sol(n, e, g) == max(b) and dfs_stack_solution(n, e, g) == b
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [24]:
# Bài 5: đếm đồ thị con liên thông chứa mỗi đỉnh
MOD = 10**9 + 7
def count_connected_sol(n, edges):
    adj, parent, order = bfs_order(n, edges, 0)
    ch = [[] for _ in range(n)]
    for u in order[1:]: ch[parent[u]].append(u)
    f = [1] * n
    for u in reversed(order):
        for c in ch[u]: f[u] = f[u] * (1 + f[c]) % MOD
    up = [0] * n                                  # số đồ thị con chứa p, không vào nhánh c (phía trên c)
    for u in order:
        k = len(ch[u]); vals = [1 + f[c] for c in ch[u]]
        pre = [1] * (k + 1); suf = [1] * (k + 1)
        for i in range(k): pre[i + 1] = pre[i] * vals[i] % MOD
        for i in range(k - 1, -1, -1): suf[i] = suf[i + 1] * vals[i] % MOD
        for i, c in enumerate(ch[u]):
            up[c] = (1 + up[u]) * pre[i] % MOD * suf[i + 1] % MOD
    return [f[v] * (1 + up[v]) % MOD for v in range(n)]

def count_brute(n, edges):
    adj = [[] for _ in range(n)]
    for a, b in edges: adj[a].append(b); adj[b].append(a)
    cnt = [0] * n
    for mask in range(1, 1 << n):
        st = (mask & -mask).bit_length() - 1; seen = 1 << st; stack = [st]
        while stack:
            u = stack.pop()
            for v in adj[u]:
                if mask >> v & 1 and not seen >> v & 1: seen |= 1 << v; stack.append(v)
        if seen == mask:
            for v in range(n):
                if mask >> v & 1: cnt[v] += 1
    return cnt

rng = random.Random(5)
for _ in range(300):
    n = rng.randint(2, 10); e = rand_tree(rng, n)
    assert count_connected_sol(n, e) == count_brute(n, e)
print("Bài 5 OK; ví dụ đường thẳng 4 đỉnh:", count_connected_sol(4, [[0, 1], [1, 2], [2, 3]]))

Bài 5 OK; ví dụ đường thẳng 4 đỉnh: [4, 6, 6, 4]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
order = BFS từ 0 (ghi parent)
down = [w]                          # w = +1 tốt / -1 xấu
for u in reversed(order): if parent[u] >= 0: down[parent[u]] += max(0, down[u])
ans[0] = down[0]
for u in order[1:]:
    p = parent[u]
    ans[u] = down[u] + max(0, ans[p] - max(0, down[u]))
```

**Checklist:**

- [ ] Đổi `good` thành `±1`?
- [ ] Pha 1 duyệt **ngược** BFS, pha 2 duyệt **xuôi**?
- [ ] Trừ `max(0, down[c])`, không phải `down[c]`?
- [ ] Không dùng đệ quy sâu?
- [ ] Đã so với vét cạn trên cây nhỏ?